In [ ]:
"""
Introduction to Aggregation
    Aggregation in Spark are implemented using functions

Comonly used aggregate functions
    count(*), count(expr), count(DISTINCT expr)
    min(expr), max(expr), avg(expr), sum(expr)
    Types of Aggregation
    Simple Aggregation
    Grouped Aggregation
    Multilevel Aggregation
    Window Aggregation
"""

In [12]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [18]:
"""
Requirement - Analysis Data Set
Prepare club bookings dataset for analysis

+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
"""

members_df = spark.table("spark_db.members")
bookings_df = spark.table("spark_db.bookings")
facilities_df = spark.table("spark_db.facilities")

club_bookings_df = (
    bookings_df.join(facilities_df, "facid")
            .join(members_df, "memid", "left")
            .selectExpr("bookid",
                        "case when memid==0 then 'Guest Member' else concat_ws(' ', firstname, surname) end as member_name",
                        "fac_name","starttime",
                        "case when memid == 0 then slots * guestcost else slots * membercost end as booking_amount")            
)

club_bookings_df.show()

+------+------------+---------------+-------------------+--------------+
|bookid| member_name|       fac_name|          starttime|booking_amount|
+------+------------+---------------+-------------------+--------------+
|     0|Darren Smith|   Table Tennis|2022-07-03 11:00:00|             0|
|     1|Darren Smith| Massage Room 1|2022-07-03 08:00:00|            70|
|     2|Guest Member|   Squash Court|2022-07-03 18:00:00|          NULL|
|     3|Darren Smith|  Snooker Table|2022-07-03 19:00:00|             0|
|     4|Darren Smith|     Pool Table|2022-07-03 10:00:00|             0|
|     5|Darren Smith|     Pool Table|2022-07-03 15:00:00|             0|
|     6| Tracy Smith| Tennis Court 1|2022-07-04 09:00:00|            15|
|     7| Tracy Smith| Tennis Court 1|2022-07-04 15:00:00|            15|
|     8|  Tim Rownam| Massage Room 1|2022-07-04 13:30:00|            70|
|     9|Guest Member| Massage Room 1|2022-07-04 15:00:00|           160|
|    10|Guest Member| Massage Room 1|2022-07-04 17:

Calculate total earnings and average booking value.

In [19]:
"""
1.1 Using sql like expressions
"""

result_df = club_bookings_df.selectExpr("sum(booking_amount) as total_earnings",
                                        "avg(booking_amount) as avg_booking_value")
result_df.show()

+--------------+-----------------+
|total_earnings|avg_booking_value|
+--------------+-----------------+
|        117210|32.87798036465638|
+--------------+-----------------+



In [22]:
"""
1.2 Using column expressions
"""
from pyspark.sql.functions import sum, avg, col # type: ignore

result_df = club_bookings_df.select(
    sum(col("booking_amount")).alias("total_earnings"),
    avg(col("booking_amount")).alias("avg_booking_value")
)
result_df.show()

+--------------+-----------------+
|total_earnings|avg_booking_value|
+--------------+-----------------+
|        117210|32.87798036465638|
+--------------+-----------------+



In [ ]:
"""
1.3 Using aggregate transformation
"""

result_df = club_bookings_df.agg(                   # agg expression is used to calculate aggregations
    sum(col("booking_amount")).alias("total_earnings"),
    avg(col("booking_amount")).alias("avg_booking_value")
)
result_df.show()

+--------------+-----------------+
|total_earnings|avg_booking_value|
+--------------+-----------------+
|        117210|32.87798036465638|
+--------------+-----------------+

